# 04 — Testing the updated algorithm on the Cardiac dataset

This notebook performs an end-to-end evaluation of the current deduplication algorithm on an unseen gold standard dataset - the Cardiac dataset from the ASySD evaluation. This dataset can be used for additional tweaks/further development before we apply to another unseen dataset for evaluation. 

## Setup

Imports cover three areas:

- **Standard data science stack** (`pandas`, `numpy`, `sklearn`) for data handling and computing evaluation metrics.
- **App modules** — `Deduper` is the main deduplication class; `BLOCK_RULES` defines which field combinations are used for candidate-pair blocking; `ExtendedPaper` is the Pydantic model that adds gold-standard fields (`recordid`, `duplicateid`) on top of the base `Paper` schema.
- **Path configuration** — the notebook resolves the repository root dynamically so it can be run from either the `notebooks/` folder or the repo root.

In [1]:
from __future__ import annotations

import re
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from pydantic import ValidationError
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from tqdm import tqdm

# Ensure imports like "from app..." work when notebook is opened from notebooks/
repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from app.dedupe import Deduper
from app.determine_weights import BLOCK_RULES, ExtendedPaper
from app.normalisers import normalize_doi, normalize_pages, strip_doi_punctuation

DATA_PATH = repo_root / "notebooks/data/cardiac_data.csv"
SCORES_OUT = repo_root / "notebooks/results/cardiac_scored_pairs.csv"
FALSE_POSITIVES_DIR = repo_root / "notebooks/results/cardiac_false_positives"


## Data loading and record cache

The Cardiac dataset (`notebooks/data/cardiac_data.csv`) contains **8,948 records** from a preclinical systematic review search, with expert-labelled duplicate groups. Each duplicate group shares a common `duplicateid`; records with a unique or missing `duplicateid` are true uniques.

Two helper functions are defined here:

- **`load_data`** — reads the CSV, normalises column names (strips BOM characters, lowercases), and renames `author`→`authors` and `number`→`issue` to match the `Paper` schema.
- **`build_record_cache`** — parses every row into an `ExtendedPaper` Pydantic object and stores them in a `dict` keyed by `recordid`. Pre-building this dictionary means each record is parsed only once, making bulk pair scoring O(1) per lookup rather than O(n) per pair.

In [2]:
def norm_value(field: str, val) -> str | None:
    if pd.isna(val) or val is None or val == "":
        return None

    text = str(val).strip()
    if text == "":
        return None

    # Field-aware normalization before blocking
    if field == "doi":
        doi = normalize_doi(text)
        return strip_doi_punctuation(doi) if doi else None

    if field == "pages":
        pages = normalize_pages(text)
        return re.sub(r"\s+", "", pages).lower() if pages else None

    if field in {"year", "volume", "issue"}:
        cleaned = re.sub(r"\s+", "", text)
        try:
            return str(int(float(cleaned)))
        except ValueError:
            return cleaned.lower()

    return re.sub(r"\s+", " ", text.lower())


def _sanitize_row_for_pydantic(record: dict) -> dict:
    sanitized = {}
    for key, value in record.items():
        if pd.isna(value):
            sanitized[key] = None
            continue

        if isinstance(value, str):
            stripped = value.strip()
            value = stripped if stripped else None

        if key == "doi" and value is not None:
            normalized = normalize_doi(str(value))
            sanitized[key] = normalized if normalized else None
        else:
            sanitized[key] = value
    return sanitized


def build_record_cache(
    df: pd.DataFrame,
    id_column: str = "recordid",
    include_ids: set[int] | None = None,
) -> dict[int, ExtendedPaper]:
    cache: dict[int, ExtendedPaper] = {}
    validation_errors = 0

    for record in df.to_dict(orient="records"):
        record_id = record.get(id_column)
        if record_id is None or pd.isna(record_id):
            continue

        parsed_id = int(record_id)
        if include_ids is not None and parsed_id not in include_ids:
            continue

        cleaned_record = _sanitize_row_for_pydantic(record)
        try:
            cache[parsed_id] = ExtendedPaper(**cleaned_record)
        except ValidationError:
            validation_errors += 1

    if validation_errors:
        print(f"Skipped {validation_errors} invalid records while building the cache")

    return cache


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="latin-1")
    df.columns = [c.replace("ï..", "").strip().lower() for c in df.columns]
    return df.rename(columns={"author": "authors", "number": "issue"})


df = load_data(DATA_PATH)
print(df.shape)
df.head()

(8948, 26)


,unnamed: 0,,authors,year,journal,doi,title,pages,volume,issue,...,label,endnote,bond,asysd,gold,duplicateid,nbond,nendnote,nasysd,ngold
0,1,1467,Przyklenk K.Kloner R. A.,1993,Basic Res Cardiol,10.1007/978-3-642-72497-8_10,"""Cardioprotection"" by ACE-inhibitors in acute ...",139-54,88 Suppl 1,NaN,...,In_database,KEEP,KEEP,KEEP,KEEP,3268,1,1,1,1
1,2,2667,Hirata T.Fukuse T.Ishikawa S.Hanaoka S.Chen Q....,2001,Transplantation,10.1097/00007890-200102150-00003,"""Chemical preconditioning"" by 3-nitropropionat...",352-9,71,3,...,In_database,KEEP,KEEP,KEEP,KEEP,1658,2,1,1,1
2,3,7774,Hirata T.Fukuse T.Ishikawa S.Hanaoka S.Chen Q....,2001,Transplantation,NaN,"""Chemical preconditioning"" by 3-nitropropionat...",352-359,71,3,...,Duplicate_in_trash,REMOVE,KEEP,REMOVE,REMOVE,1658,2,1,1,1
3,4,3129,Lawson C. S.Coltart D. J.Hearse D. J.,1993,J Mol Cell Cardiol,10.1006/jmcc.1993.1156,"""Dose""-dependency and temporal characteristics...",1391-402,25,12,...,In_database,KEEP,KEEP,KEEP,KEEP,2255,2,1,1,1
4,5,2258,Dickson E. W.Porcaro W. A.Fenton R. A.Heard S....,2000,Acad Emerg Med,10.1111/j.1553-2712.2000.tb02228.x,"""Preconditioning at a distance"" in the isolate...",311-7,7,4,...,In_database,KEEP,KEEP,KEEP,KEEP,1059,2,1,1,1


## Blocking: generating candidate pairs

We use the updated blocking rules to generate comparisons. The rules are:

| Rule fields | Rationale |
|---|---|
| `title` | Records sharing a normalised title token are very likely duplicates |
| `abstract` | Shared abstract text is a strong duplicate signal |
| `doi` | Exact (normalised) DOI match |
| `year` + `journal` | Same year and journal context |
| `year` + `pages` | Same year and page range |
| `year` + `volume` | Same year and volume |

Before grouping, values are normalised with field-aware logic:

- DOI: canonical DOI extraction plus punctuation-stripped suffix fallback
- Pages: canonical page-range normalisation
- Year/Volume/Issue: numeric canonicalisation (e.g. `01` -> `1`)
- Other fields: lowercase and collapsed whitespace

Pairs caught by multiple rules are deduplicated so each `(id_a, id_b)` appears exactly once in the output, but all matched rules are recorded in `block_rules` for later inspection.

In [3]:
def build_blocked_pairs(df: pd.DataFrame) -> pd.DataFrame:
    dup_lookup = df.set_index("recordid")["duplicateid"].to_dict()
    seen = set()
    rows = []
    pair_rules: dict[tuple, set[str]] = {}

    for rule in BLOCK_RULES:
        missing = [field for field in rule if field not in df.columns]
        if missing:
            continue

        rule_label = ",".join(rule)
        subset = df[["recordid", *rule]].copy()
        norm_cols = []
        for field in rule:
            norm_col = f"{field}_norm"
            subset[norm_col] = subset[field].apply(
                lambda v, field_name=field: norm_value(field_name, v)
            )
            norm_cols.append(norm_col)

        subset = subset.dropna(subset=norm_cols)
        subset["block_key"] = subset[norm_cols].apply(tuple, axis=1)

        for _, group in subset.groupby("block_key"):
            ids = group["recordid"].tolist()
            if len(ids) < 2:
                continue

            for a, b in combinations(ids, 2):
                id_a, id_b = (a, b) if a < b else (b, a)
                key = (id_a, id_b)
                pair_rules.setdefault(key, set()).add(rule_label)
                if key in seen:
                    continue

                dup_a = dup_lookup.get(id_a)
                dup_b = dup_lookup.get(id_b)
                is_dupe = int(pd.notna(dup_a) and pd.notna(dup_b) and dup_a == dup_b)

                rows.append((id_a, id_b, is_dupe))
                seen.add(key)

    all_pairs_df = pd.DataFrame(rows, columns=["id_a", "id_b", "is_dupe"])
    all_pairs_df["block_rules"] = all_pairs_df.apply(
        lambda r: " | ".join(sorted(pair_rules.get((r["id_a"], r["id_b"]), set()))), axis=1
    )
    return all_pairs_df


all_pairs_df = build_blocked_pairs(df)

# Overall totals
n_dupes = all_pairs_df["is_dupe"].sum()
n_non_dupes = (all_pairs_df["is_dupe"] == 0).sum()
print(f"Total pairs: {len(all_pairs_df):,}  |  dupes: {n_dupes:,}  |  non-dupes: {n_non_dupes:,}")

Total pairs: 62,197  |  dupes: 3,821  |  non-dupes: 58,376


## Pair selection

`MAX_PAIRS` controls whether to work with all blocked pairs or a random subsample. Set it to `None` (default) to score every pair; set it to a fixed integer to cap the workload during development or quick iteration. When a subsample is used, the gold-standard `is_dupe` labels are preserved, so all downstream metrics remain valid — they just reflect a subset of the full candidate space.

In [4]:
MAX_PAIRS = None  # set to None to use all blocked pairs

if MAX_PAIRS is not None and len(all_pairs_df) > MAX_PAIRS:
    pairs_df = all_pairs_df.sample(n=MAX_PAIRS, random_state=1234).reset_index(drop=True)
else:
    pairs_df = all_pairs_df.copy()

print(pairs_df.shape)
pairs_df.head()


(62197, 4)


,id_a,id_b,is_dupe,block_rules
0,1658,7880,1,"pages,volume | title | year,journal | year,pag..."
1,3116,7406,1,"doi | pages,volume | title | year,journal | ye..."
2,409,6912,1,"abstract | doi | pages,volume | title | year,j..."
3,332,7071,1,"doi | pages,volume | title | year,journal | ye..."
4,4185,6779,1,"doi | pages,volume | title | year,journal | ye..."


## Scoring pairs

Each candidate pair is scored using `Deduper.dedupe_weighted()`, which applies the logistic regression model derived in `01_building_dedupe_feature_importance_model.ipynb`:

Again, we are excluding **`issue` and `abstract`** from the weighted sum here — both have negative or near-zero weights in the fitted model and were found to add noise rather than signal when used in the scoring step. 

Scores are written to `notebooks/results/cardiac_false_positives.csv` for reuse across evaluation runs.

In [5]:
import importlib
import app.dedupe as _dedupe_mod
importlib.reload(_dedupe_mod)

from loguru import logger
from app.dedupe import Deduper, WEIGHTS, INTERCEPT

# Weights excluding 'issue' and 'abstract'
WEIGHTS_FILTERED = {k: v for k, v in WEIGHTS.items() if k not in ("issue", "abstract")}

SCORE_FIELDS = list(WEIGHTS_FILTERED.keys())  # fields whose individual scores will be exported

def score_pairs_weighted(df: pd.DataFrame, pairs_df: pd.DataFrame) -> pd.DataFrame:
    """Score pairs using Deduper's weighted dedupe logic, excluding 'issue' and 'abstract'."""
    record_cache = build_record_cache(df, id_column="recordid")

    any_paper = next(iter(record_cache.values()))
    deduper = Deduper(reference=any_paper, candidates=[any_paper])

    results = []
    logger.disable("app.dedupe")
    try:
        for row in tqdm(pairs_df.itertuples(index=False), total=len(pairs_df)):
            rec_a = record_cache.get(int(row.id_a))
            rec_b = record_cache.get(int(row.id_b))
            if rec_a is None or rec_b is None:
                continue
            prob, field_scores, early_stop = deduper.score_pair(
                rec_a, rec_b, weights=WEIGHTS_FILTERED, intercept=INTERCEPT
            )
            result = {
                "id_a": row.id_a,
                "id_b": row.id_b,
                "is_dupe": row.is_dupe,
                "prob": prob,
                "early_stop": early_stop,
            }
            for field in SCORE_FIELDS:
                result[f"score_{field}"] = field_scores.get(field)
            results.append(result)
    finally:
        logger.enable("app.dedupe")

    return pd.DataFrame(results)

scored_df = score_pairs_weighted(df, pairs_df)
scored_df.head()


Skipped 8 invalid records while building the cache


100%|██████████| 62197/62197 [00:02<00:00, 22736.64it/s]


,id_a,id_b,is_dupe,prob,early_stop,score_doi,score_title,score_authors,score_year,score_journal,score_pages,score_volume
0,1658,7880,1,0.950263,None,NaN,1.0,1.000000,1.0,1.0,1.0,1.0
1,3116,7406,1,0.994048,None,1.0,1.0,0.954135,1.0,1.0,1.0,1.0
2,409,6912,1,0.994675,None,1.0,1.0,1.000000,1.0,1.0,1.0,1.0
3,332,7071,1,0.994675,None,1.0,1.0,1.000000,1.0,1.0,1.0,1.0
4,4185,6779,1,0.994675,None,1.0,1.0,1.000000,1.0,1.0,1.0,1.0


## Pair-level evaluation across thresholds

For each score threshold in `THRESHOLDS`, a pair is classified as a predicted duplicate if `prob >= threshold`. We compute:

| Metric | Definition |
|---|---|
| **Sensitivity** (recall) | $\frac{TP}{TP + FN}$ — fraction of true duplicate pairs detected |
| **Specificity** | $\frac{TN}{TN + FP}$ — fraction of non-duplicate pairs correctly rejected |
| **Precision** | $\frac{TP}{TP + FP}$ — fraction of predicted duplicates that are genuine |
| **ROC-AUC** | Threshold-agnostic ranking quality |
| **Average precision** | Area under the precision-recall curve |

False positive pairs are also exported to `notebooks/results/cardiac_dataset_run/false_positives_{threshold}.csv` with both records shown side-by-side, to support qualitative inspection and inform further rule development.


In [6]:
THRESHOLDS = [0.7, 0.75, 0.8, 0.85, 0.90]

def metrics_for_threshold(scored_df: pd.DataFrame, threshold: float) -> dict:
    y_true = scored_df["is_dupe"].astype(int)
    y_prob = scored_df["prob"].astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0

    return {
        "threshold": threshold,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "average_precision": float(average_precision_score(y_true, y_prob)),
    }

import os

def _pair_side_by_side_export(df, pair_df, fields, out_path):
    # Metadata columns to carry through from scored_df
    meta_cols = [c for c in ["prob", "early_stop"] + [f"score_{f}" for f in SCORE_FIELDS] if c in pair_df.columns]

    left = df[["recordid"] + fields].copy()
    left.columns = ["id_a"] + [f"{f}_a" for f in fields]
    right = df[["recordid"] + fields].copy()
    right.columns = ["id_b"] + [f"{f}_b" for f in fields]

    export = pair_df[["id_a", "id_b"] + meta_cols].merge(left, on="id_a", how="left").merge(right, on="id_b", how="left")

    # Side-by-side field pairs first, then scores
    side_by_side = ["id_a", "id_b"] + meta_cols + [
        col
        for f in fields
        for col in (f"{f}_a", f"{f}_b")
        if col in export.columns
    ]
    export_subset = export[[c for c in side_by_side if c in export.columns]]
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    export_subset.to_csv(out_path, index=False)
    return out_path

def export_false_positives_side_by_side(df, scored_df, fields, threshold, out_path):
    y_true = scored_df["is_dupe"].astype(int)
    y_pred = (scored_df["prob"].astype(float) >= threshold).astype(int)
    fp_df = scored_df[(y_true == 0) & (y_pred == 1)].copy()
    return _pair_side_by_side_export(df, fp_df, fields, out_path)

def export_false_negatives_side_by_side(df, scored_df, fields, threshold, out_path):
    y_true = scored_df["is_dupe"].astype(int)
    y_pred = (scored_df["prob"].astype(float) >= threshold).astype(int)
    fn_df = scored_df[(y_true == 1) & (y_pred == 0)].copy()
    return _pair_side_by_side_export(df, fn_df, fields, out_path)

metrics = []
for threshold in THRESHOLDS:
    row = metrics_for_threshold(scored_df, threshold)
    metrics.append(row)
    export_false_positives_side_by_side(df, scored_df, SCORE_FIELDS, threshold, f"results/cardiac_dataset_run/false_positives_{threshold}.csv")
    export_false_negatives_side_by_side(df, scored_df, SCORE_FIELDS, threshold, f"results/cardiac_dataset_run/false_negatives_{threshold}.csv")

metrics_df = pd.DataFrame(metrics)
metrics_df


,threshold,tp,fp,tn,fn,sensitivity,specificity,precision,roc_auc,average_precision
0,0.70,3803,9,58288,13,0.996593,0.999846,0.997639,0.99895,0.998013
1,0.75,3802,6,58291,14,0.996331,0.999897,0.998424,0.99895,0.998013
2,0.80,3799,3,58294,17,0.995545,0.999949,0.999211,0.99895,0.998013
3,0.85,3776,1,58296,40,0.989518,0.999983,0.999735,0.99895,0.998013
4,0.90,3688,0,58297,128,0.966457,1.000000,1.000000,0.99895,0.998013


## Selecting the operating threshold

For deduplication in systematic review pipelines, the primary concern is **not discarding genuine unique records** (i.e. minimising false positives at the record level). We therefore search for the threshold that maximises **specificity** subject to a minimum sensitivity constraint of 98% — meaning at most 2% of true duplicates are allowed to survive.

The `find_best_threshold` function scans the metrics table and returns the threshold that best satisfies this trade-off. If no threshold meets the minimum sensitivity requirement, it reports this explicitly so the threshold can be reconsidered.

In [7]:
def find_best_threshold(metrics_df, min_sensitivity=0.99):
    """
    Find the threshold with the highest specificity where sensitivity >= min_sensitivity.
    Returns a dict with the best row, or None if no threshold meets the criteria.
    """
    filtered = metrics_df[metrics_df['sensitivity'] >= min_sensitivity]
    if filtered.empty:
        return None
    best_row = filtered.loc[filtered['specificity'].idxmax()]
    return best_row

# Example usage after metrics_df is created:
best = find_best_threshold(metrics_df, min_sensitivity=0.99)
if best is not None:
    print(f"Best threshold with sensitivity >= 0.99: {best['threshold']:.3f}")
    print(best)
else:
    print("No threshold found with sensitivity >= 0.99")


Best threshold with sensitivity >= 0.99: 0.800
threshold                0.800000
tp                    3799.000000
fp                       3.000000
tn                   58294.000000
fn                      17.000000
sensitivity              0.995545
specificity              0.999949
precision                0.999211
roc_auc                  0.998950
average_precision        0.998013
Name: 2, dtype: float64


## Record-level evaluation (ASySD-style)

The pair-level metrics above tell us how well the model ranks pairs, but they do not directly answer the question a researcher cares about: *which records will I end up with after deduplication?*

We adopt the evaluation protocol from Hair et al. (2023) used in the [ASySD R package](https://github.com/camaradesuk/ASySD):

1. **Cluster** — scored pairs above the threshold are treated as edges in a graph; connected components become predicted duplicate groups.
2. **Retain one per cluster** — within each predicted group, the record with the lowest `recordid` is kept and all others are marked as removed. (The choice of which record to retain does not affect the confusion matrix.)
3. **Gold standard** — records sharing a `duplicateid` form a true duplicate group; one record per group should be kept and the rest should be removed. Records with a unique or missing `duplicateid` are true uniques and should always be kept.
4. **Confusion matrix** at the record level:

| | **Predicted: removed** | **Predicted: kept** |
|---|---|---|
| **True: duplicate** | TP — correctly removed | FN — missed duplicate |
| **True: unique** | FP — wrongly removed | TN — correctly kept |

The key metrics are:
- **Sensitivity** $= \frac{TP}{TP+FN}$ — proportion of true duplicates that were successfully removed.
- **Specificity** $= \frac{TN}{TN+FP}$ — proportion of true unique records that were correctly retained. A single false positive here means a real paper was lost from the search results.
- **Precision** $= \frac{TP}{TP+FP}$ — of all records removed, what fraction were genuine duplicates.

In [8]:
import networkx as nx

THRESHOLD = 0.80  # adjust as needed

# --- Step 1: Cluster positive pairs into connected components ---
def cluster_records(scored_df: pd.DataFrame, threshold: float, all_record_ids: set) -> pd.Series:
    """Return a Series mapping recordid -> predicted_group (integer)."""
    pos_pairs = scored_df[scored_df['prob'] >= threshold][['id_a', 'id_b']].to_numpy()
    G = nx.Graph()
    G.add_nodes_from(all_record_ids)
    G.add_edges_from(pos_pairs)
    record_to_group = {}
    for group_id, component in enumerate(nx.connected_components(G)):
        for rid in component:
            record_to_group[rid] = group_id
    return pd.Series(record_to_group, name='predicted_group')

all_record_ids = set(df['recordid'])
record_to_group = cluster_records(scored_df, THRESHOLD, all_record_ids)
df['predicted_group'] = df['recordid'].map(record_to_group)

# For each predicted cluster, retain the record with the lowest recordid (ASySD convention)
df['pred_keep'] = df.groupby('predicted_group')['recordid'].transform('min') == df['recordid']

print(f"Total records: {len(df)}")
print(f"Predicted clusters: {df['predicted_group'].nunique()}")
print(f"Records retained (pred_keep): {df['pred_keep'].sum()}")
print(f"Records removed: {(~df['pred_keep']).sum()}")
df[['recordid', 'predicted_group', 'pred_keep']].head(10)

Total records: 8948
Predicted clusters: 5434
Records retained (pred_keep): 5434
Records removed: 3514


,recordid,predicted_group,pred_keep
0,3268,2989,True
1,1658,1380,True
2,7880,1380,False
3,2255,1977,True
4,1059,781,True
5,5851,4943,True
6,3116,2837,True
7,7406,2837,False
8,8357,5353,True
9,332,54,True


### Step 1: Build gold-standard labels

The gold standard is derived directly from the `duplicateid` column:

- Records that share a `duplicateid` belong to the same true duplicate group. Exactly one record per group (the one with the lowest `recordid`) is designated as the **true keep**; all others are **true duplicates that should be removed**.
- Records with a missing `duplicateid` are singletons — true uniques — and are always designated as **true keeps**.

In [9]:
# --- Step 2: Gold standard labels ---
# Records sharing a duplicateid are a true duplicate group; one per group should be kept.
# Records with a unique/missing duplicateid are true uniques and should be kept.

# Mark exactly one record per duplicateid group as the 'true keep'
df['true_cluster'] = df['duplicateid'].fillna(df['recordid'].astype(str)).astype(str)
df['true_keep'] = df.groupby('true_cluster')['recordid'].transform('min') == df['recordid']

n_true_dupes = (~df['true_keep']).sum()
n_true_unique = df['true_keep'].sum()
print(f"Gold standard: {n_true_unique} unique records to keep, {n_true_dupes} duplicates to remove")

Gold standard: 5418 unique records to keep, 3530 duplicates to remove


### Step 2: Compute the record-level confusion matrix

Having established both `true_removed` (gold standard) and `pred_removed` (algorithm output) as boolean columns on every record, the four cells of the confusion matrix follow directly from their intersection. The printed output shows absolute counts alongside sensitivity, specificity, and precision so the trade-off at the chosen threshold is immediately readable.

In [10]:
# --- Step 3: Record-level confusion matrix (ASySD definitions) ---
# true_removed = citation IS a duplicate and should have been removed
# pred_removed = citation WAS removed by the algorithm

df['true_removed'] = ~df['true_keep']
df['pred_removed'] = ~df['pred_keep']

TP = int(( df['true_removed'] &  df['pred_removed']).sum())  # duplicates correctly removed
FP = int((~df['true_removed'] &  df['pred_removed']).sum())  # uniques wrongly removed
TN = int((~df['true_removed'] & ~df['pred_removed']).sum())  # uniques correctly kept
FN = int(( df['true_removed'] & ~df['pred_removed']).sum())  # duplicates wrongly kept

sensitivity = TP / (TP + FN) if (TP + FN) else 0.0  # recall: fraction of dupes found
specificity = TN / (TN + FP) if (TN + FP) else 0.0  # fraction of uniques kept
precision   = TP / (TP + FP) if (TP + FP) else 0.0  # fraction removed that were real dupes

print(f"Threshold: {THRESHOLD}")
print(f"TP (duplicates correctly removed): {TP}")
print(f"FP (unique records wrongly removed): {FP}")
print(f"TN (unique records correctly kept): {TN}")
print(f"FN (duplicates missed / wrongly kept): {FN}")
print()
print(f"Sensitivity (recall): {sensitivity:.4f}  — % of true duplicates removed")
print(f"Specificity:          {specificity:.4f}  — % of true uniques kept")
print(f"Precision:            {precision:.4f}  — % of removed records that were real duplicates")

Threshold: 0.8
TP (duplicates correctly removed): 3511
FP (unique records wrongly removed): 3
TN (unique records correctly kept): 5415
FN (duplicates missed / wrongly kept): 19

Sensitivity (recall): 0.9946  — % of true duplicates removed
Specificity:          0.9994  — % of true uniques kept
Precision:            0.9991  — % of removed records that were real duplicates


In [11]:
# --- Sweep across thresholds: record-level confusion matrix for each ---
import pandas as pd


def record_level_metrics_for_threshold(df_orig, scored_df, threshold):
    df_t = df_orig.copy()
    valid_ids = set(df_t["recordid"].astype(int))

    # Guard against stale scored_df from a different dataset/kernel state.
    scored_aligned = scored_df[
        scored_df["id_a"].astype(int).isin(valid_ids)
        & scored_df["id_b"].astype(int).isin(valid_ids)
    ].copy()

    if scored_aligned.empty:
        raise ValueError(
            "No scored pairs overlap with the current dataset record IDs. "
            "Re-run the scoring cell for this dataset before record-level evaluation."
        )

    # Cluster
    G = nx.Graph()
    G.add_nodes_from(valid_ids)
    pos_pairs = scored_aligned[scored_aligned["prob"] >= threshold][["id_a", "id_b"]].to_numpy()
    G.add_edges_from(pos_pairs)

    record_to_group = {}
    for gid, comp in enumerate(nx.connected_components(G)):
        for rid in comp:
            record_to_group[int(rid)] = gid

    df_t["recordid"] = df_t["recordid"].astype(int)
    df_t["predicted_group"] = df_t["recordid"].map(record_to_group)
    df_t["pred_keep"] = (
        df_t.groupby("predicted_group")["recordid"].transform("min") == df_t["recordid"]
    )

    # Gold standard
    df_t["true_cluster"] = df_t["duplicateid"].fillna(df_t["recordid"].astype(str)).astype(str)
    df_t["true_keep"] = (
        df_t.groupby("true_cluster")["recordid"].transform("min") == df_t["recordid"]
    )

    # Confusion matrix
    true_rem = ~df_t["true_keep"]
    pred_rem = ~df_t["pred_keep"]
    tp = int(( true_rem &  pred_rem).sum())
    fp = int((~true_rem &  pred_rem).sum())
    tn = int((~true_rem & ~pred_rem).sum())
    fn = int(( true_rem & ~pred_rem).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    return {
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "sensitivity": sens,
        "specificity": spec,
        "precision": prec,
        "pairs_used": int(len(scored_aligned)),
    }


# quick sanity checks before record-level sweep
all_ids = set(load_data(DATA_PATH)["recordid"].astype(int))
overlap_mask = scored_df["id_a"].astype(int).isin(all_ids) & scored_df["id_b"].astype(int).isin(all_ids)
print(f"Pairs in scored_df: {len(scored_df):,}")
print(f"Pairs overlapping current dataset IDs: {int(overlap_mask.sum()):,}")

if MAX_PAIRS is not None:
    print("Warning: MAX_PAIRS is set, so record-level metrics are based on sampled pairs and can underestimate recall.")


df_base = load_data(DATA_PATH)  # fresh copy without computed columns
record_metrics = [record_level_metrics_for_threshold(df_base, scored_df, t) for t in THRESHOLDS]
record_metrics_df = pd.DataFrame(record_metrics)
record_metrics_df

Pairs in scored_df: 62,113
Pairs overlapping current dataset IDs: 62,113


,threshold,TP,FP,TN,FN,sensitivity,specificity,precision,pairs_used
0,0.70,3515,7,5411,15,0.995751,0.998708,0.998012,62113
1,0.75,3514,5,5413,16,0.995467,0.999077,0.998579,62113
2,0.80,3511,3,5415,19,0.994618,0.999446,0.999146,62113
3,0.85,3492,1,5417,38,0.989235,0.999815,0.999714,62113
4,0.90,3410,0,5418,120,0.966006,1.000000,1.000000,62113


### Step 3: Threshold sweep at the record level

The single-threshold evaluation above gives a snapshot, but the optimal operating point may differ when measured at the record level versus the pair level (because clustering can propagate errors across records). This sweep reruns the full cluster → keep/remove pipeline for every threshold in `THRESHOLDS` and reports the record-level confusion matrix for each.

The resulting `record_metrics_df` table is the primary output for deciding which threshold to carry forward into production or unseen-dataset testing. A threshold that looks good at the pair level may look worse here if it causes aggressive over-clustering (many FPs) or under-clustering (many FNs).